In [ ]:
pip install faiss-cpu sentence-transformers pandas
# Install required libraries for vector search and embeddings

In [ ]:
import pandas as pd

# Load the dataset from Google Drive
# The dataset contains two main columns: 'question' and 'answer'
# This is the normalized dataset prepared during data cleaning phase

df = pd.read_csv("/content/drive/MyDrive/llama model/normalized_ds_questions.csv")


# Convert DataFrame columns into Python lists
# These lists will be used for model training or processing
questions = df["question"].tolist()
answers = df["answer"].tolist()

In [ ]:
from sentence_transformers import SentenceTransformer


# Load lightweight pre-trained embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
 

# Convert questions into vector embeddings
question_embeddings = embed_model.encode(questions)

In [ ]:

# This step builds a FAISS vector index for efficient similarity search on embeddings.
# It enables fast retrieval of relevant questions based on distance metrics.

import faiss
import numpy as np

# Get embedding dimension (number of features per vector)
dimension = question_embeddings.shape[1]

# Initialize FAISS index with L2 (Euclidean) distance
index = faiss.IndexFlatL2(dimension)

# Add all question embeddings into the index
index.add(np.array(question_embeddings))

# Confirm index is built by printing total stored vectors
print("FAISS index ready:", index.ntotal)

FAISS index ready: 1326


In [ ]:
# This function retrieves top-k most similar question-answer pairs using vector similarity search.
# It encodes the query, searches the FAISS index, and returns matched results.

def retrieve(query, k=3):
    # Convert query into embedding vector
    query_vec = embed_model.encode([query])

    # Search for top-k closest vectors in FAISS index
 
    distances,indices = index.search(query_vec, k)

    results = []
    # Map retrieved indices to original questions and answers
    for i in indices[0]:
        results.append((questions[i], answers[i]))

    return results

In [ ]:
def build_prompt(query, retrieved_docs,answer):
    context = "\n".join([f"Q: {q}\nA: {a}" for q, a in retrieved_docs])

    prompt = f"""
You are an AI interview evaluator.

Context:
{context}

candidates answer:
{answer}

Question:
{query}

  Give output in this format ONLY:
Evaluation:
...
Suggestion:
...
Score: <number>/100
"""
    return prompt

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_answer(query,answer):
    retrieved = retrieve(query)
    prompt = build_prompt(query, retrieved,answer)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.3,
        do_sample=True,
    )

    full_output= tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_output.replace(prompt, "").strip()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)



In [2]:
def evaluate_answer(answer):
    prompt = f"""
You are a strict technical interview evaluator.

Evaluate the candidate's answer based on:
- Clarity
- Accuracy
- Depth

Answer:
{answer}

Return in this format:

Score: X/10
Clarity: Good/Average/Poor
Accuracy: Good/Average/Poor
Depth: Good/Average/Poor
Feedback:
Improvements:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3,   # lower = more strict/deterministic
        top_p=0.9
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [14]:
def evaluate_answer(answer,question):
    prompt = f"""
You are a strict technical interview evaluator.

Evaluate the candidate's answer.
Question:
{question}
Answer:
{answer}

Return evaluation with suggestion and correct answer also give metrics in numbers
Score: X/10
Clarity:
Accuracy::
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.3
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove prompt part
    return full_output.replace(prompt, "").strip()

In [15]:
questions = [
    # Machine Learning / AI (20)
    "What is overfitting?",
    "What is underfitting?",
    "Explain bias vs variance tradeoff",
    "What is gradient descent?",
    "What is stochastic gradient descent?",
    "What is regularization? (L1 vs L2)",
    "What is cross-validation?",
    "What is a confusion matrix?",
    "What is precision vs recall?",
    "What is F1 score?",
    "What is ROC-AUC?",
    "Difference between classification and regression",
    "What is a decision tree?",
    "What is a random forest?",
    "What is boosting?",
    "What is bagging?",
    "What is KNN algorithm?",
    "What is SVM?",
    "What is PCA?",
    "What is feature engineering?",

    # Statistics (10)
    "What is mean, median, and mode?",
    "What is standard deviation?",
    "What is variance?",
    "What is normal distribution?",
    "What is central limit theorem?",
    "What is hypothesis testing?",
    "What is p-value?",
    "What is confidence interval?",
    "What is correlation vs covariance?",
    "What is skewness and kurtosis?",

    # Python (10)
    "What is list vs tuple?",
    "What is dictionary in Python?",
    "What is list comprehension?",
    "What are lambda functions?",
    "What is a generator?",
    "What is difference between deep copy and shallow copy?",
    "What is exception handling?",
    "What is pandas DataFrame?",
    "What is NumPy array?",
    "What is the difference between append() and extend()?",

    # SQL (5)
    "What is SQL JOIN?",
    "What are types of joins in SQL?",
    "Difference between WHERE and HAVING",
    "What is GROUP BY?",
    "What is indexing in SQL?",

    # Case-Based / Scenario (5)
    "How would you handle missing data in a dataset?",
    "How would you detect outliers?",
    "How would you evaluate a classification model?",
    "How would you improve model performance?",
    "How would you explain a model to a non-technical person?"
]
import random

question = random.choice(questions)

print("Question:", question)

answer = input("\nYour Answer: ")


print(evaluate_answer(answer,question))

Question: What is feature engineering?

Your Answer: to extract features from raw data


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Relevance:

Suggestion:
The candidate's answer is concise and to the point. However, it lacks clarity and accuracy. The candidate could have provided a more detailed explanation of what feature engineering is and how it is used in data analysis. Additionally, the candidate's answer is not relevant to the question asked.

Correct answer:
Feature engineering is the process of selecting, transforming, and creating relevant features from raw data to improve the performance of machine learning models.

Metrics:
Clarity: 5/10
Accuracy: 3/10
Relevance: 2/10

Overall score: 3/10
